In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------

# Illustrations 9.4-2 and 9.4-3 &mdash; species fugacity from the Peng&ndash;Robinson EOS

This notebook reproduces **Illustrations 9.4-2 and 9.4-3** for an ethane / *n*-butane
mixture.

The fugacity of a species *in a mixture* is what every later phase-equilibrium
calculation needs. For the Peng&ndash;Robinson equation of state with the van der Waals
one-fluid mixing rules it is Eq. 9.4-9,

$$\ln\bar\phi_i = \frac{b_i}{b}(Z-1) - \ln(Z-B)
 - \frac{A}{2\sqrt2 B}\left[\frac{2\sum_j y_j a_{ij}}{a} - \frac{b_i}{b}\right]
   \ln\!\left[\frac{Z + (1+\sqrt2)B}{Z + (1-\sqrt2)B}\right]$$

with $a_{ij} = (1-k_{ij})\sqrt{a_i a_j}$ and $a = \sum_i\sum_j y_i y_j a_{ij}$,
$b = \sum_i y_i b_i$ (Eqs. 9.4-8).

**The binary interaction parameter is the only mixture input**, and Illustration 9.4-3
takes it from Table 9.4-1: $k_{\rm ET-BU} = 0.010$. That table is now
`code/data/pr_kij.csv`, so it can be looked up rather than typed.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst  
August 2026

In [2]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants
from thermo.charts import use_book_style
use_book_style()
R = constants.R

In [3]:
from thermo import PRMixture
from thermo.data import pr_kij_matrix

kij, missing = pr_kij_matrix(["ethane", "n-butane"])
print(f"Table 9.4-1: k_ij(ethane, n-butane) = {kij[0,1]}     (Illustration 9.4-3 uses 0.010)")
print("pairs the table does not give:", missing or "none")

m = PRMixture.from_database(["ethane", "n-butane"], kij="table")
for c in m.components:
    print(f"  {c.name:10s} Tc = {c.Tc:6.1f} K   Pc = {c.Pc/1e5:5.2f} bar   omega = {c.omega:.3f}")

Table 9.4-1: k_ij(ethane, n-butane) = 0.01     (Illustration 9.4-3 uses 0.010)
pairs the table does not give: none
  ethane     Tc =  305.4 K   Pc = 48.80 bar   omega = 0.099
  n-butane   Tc =  425.2 K   Pc = 38.00 bar   omega = 0.199


## Illustration 9.4-3 — an equimolar mixture at 373.15 K

In [4]:
T = 373.15
y = np.array([0.5, 0.5])
book = {1: (0.991, 0.498, 0.493), 10: (0.910, 4.836, 4.333), 15: (0.861, 7.143, 6.024)}

print(f"{'P (bar)':>8}{'Z':>8}{'f_ET':>9}{'f_BU':>9}   |   book Z, f_ET, f_BU")
for P_bar in (1, 10, 15):
    P = P_bar * 1e5
    Z = m.Z(y, T, P, "vapor")
    f = m.fugacity(y, T, P, "vapor") / 1e5
    b = book[P_bar]
    print(f"{P_bar:8d}{Z:8.3f}{f[0]:9.3f}{f[1]:9.3f}   |   {b[0]:.3f}, {b[1]:.3f}, {b[2]:.3f}")

 P (bar)       Z     f_ET     f_BU   |   book Z, f_ET, f_BU
       1   0.991    0.498    0.493   |   0.991, 0.498, 0.493
      10   0.910    4.836    4.333   |   0.910, 4.836, 4.333
      15   0.861    7.144    6.022   |   0.861, 7.143, 6.024


The printed table is reproduced — exactly at 1 and 10 bar, and within one unit in
the last printed digit at 15 bar.

## What $k_{ij}$ actually buys

Illustration 9.4-3 quotes one value of $k_{ij}$ and moves on. It is worth seeing how much
of the answer depends on it, because a pair for which Table 9.4-1 is blank has to be
guessed.

In [5]:
P = 15e5
for k in (0.0, 0.010, 0.05):
    mk = PRMixture.from_database(["ethane", "n-butane"], kij=[[0, k], [k, 0]])
    f = mk.fugacity(y, T, P, "vapor") / 1e5
    print(f"  k_ij = {k:5.3f}   f_ET = {f[0]:.3f} bar   f_BU = {f[1]:.3f} bar")

  k_ij = 0.000   f_ET = 7.138 bar   f_BU = 6.017 bar
  k_ij = 0.010   f_ET = 7.144 bar   f_BU = 6.022 bar
  k_ij = 0.050   f_ET = 7.165 bar   f_BU = 6.044 bar


At these conditions the sensitivity is mild — a few tenths of a percent between
$k_{ij}=0$ and $k_{ij}=0.010$. It is not always mild: Illustration 9.4-4, in the next
notebook, is at 500 bar, where the same parameter moves the answer by 2%.

**Table 9.4-1 is about 65% blank**, and its own footnote says that where a value is
missing you should *"use estimates from mixtures of similar compounds."* That is a
judgment, so `PRMixture.from_database(..., kij="table")` warns instead of silently
filling in a zero:

In [6]:
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    PRMixture.from_database(["benzene", "toluene"], kij="table")
    for warning in w:
        print("WARNING:", warning.message)

**Your turn.** Illustration 9.4-2 asks for the fugacity of each species in a
liquid mixture rather than a vapor. Repeat the 15 bar calculation with
`phase="liquid"` and check that the two fugacities are *not* equal to the vapor
values — then find the pressure at which they are, which is the bubble point.
`m.bubble_pressure(y, T)` will tell you whether you found it.